# RAG Integration Demo

This notebook demonstrates the **rag_integration** mloda plugin — an evaluation and ingestion pipeline for text and image retrieval.

What you'll see:
1. **Text retrieval** (BeIR/SciFact) — embed scientific papers + queries, measure Recall@K
2. **Full production pipeline** — chunk → deduplicate → embed → FAISS index → evaluate, all wired through mloda
3. **Multimodal retrieval** (Flickr30K + CLIP) — find images from text captions using cross-modal embeddings

**Prerequisites**
```bash
uv sync --all-extras
pip install faiss-cpu  # if not installed
```
Download datasets: use `docs/download_datasets.ipynb` or the BEIR downloader.

In [1]:
# ── Configure paths (only cell you need to edit) ──────────────────────────────
SCIFACT_DIR = "/PathToYourData/mlodadatasetevaluation/datasets/scifact"
FLICKR_DIR  = "/PathToYourData/mlodadatasetevaluation/datasets/flickr30k_raw"
HF_CACHE    = "/PathToYourData/huggingface-cache"

import os
os.environ["HF_HOME"] = HF_CACHE
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"   # macOS: prevents FAISS/PyTorch OpenMP crash
os.environ["OMP_NUM_THREADS"] = "1"

print("Paths configured.")

Paths configured.


---
## 1 — Text Retrieval: BeIR/SciFact

SciFact is a scientific fact-checking dataset from the BEIR benchmark: 5,183 corpus documents and 300 test queries with relevance labels.

**Direct pipeline:** load → embed with `sentence-transformers` → rank by cosine similarity → Recall@K.

In [2]:
from mloda.user import Options
from rag_integration.feature_groups.datasets.text.scifact import ScifactDatasetSource

options = Options(context={ScifactDatasetSource.DATA_DIR: SCIFACT_DIR})
rows = ScifactDatasetSource._load_dataset(options)

corpus  = [r for r in rows if r["row_type"] == "corpus"]
queries = [r for r in rows if r["row_type"] == "query"]
print(f"Corpus: {len(corpus)} docs  |  Queries: {len(queries)}")
print("\nSample corpus row:")
print({k: v for k, v in corpus[0].items() if k != "text"})
print("\nSample query row:")
print({k: v for k, v in queries[0].items() if k not in ("text", "relevance_scores")})

/opt/miniconda3/envs/mloda/lib/python3.12/site-packages/beir/datasets/data_loader.py:8: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


  0%|          | 0/5183 [00:00<?, ?it/s]

Corpus: 5183 docs  |  Queries: 300

Sample corpus row:
{'doc_id': '4983', 'row_type': 'corpus'}

Sample query row:
{'doc_id': '1', 'row_type': 'query', 'relevant_doc_ids': ['31715818']}


In [3]:
from rag_integration.feature_groups.rag_pipeline.embedding.sentence_transformer import SentenceTransformerEmbedder
from rag_integration.feature_groups.evaluation.metrics import mean_recall_at_k
import numpy as np

# Embed corpus + queries in one batch
all_rows = corpus + queries
texts = [r["text"] for r in all_rows]

print(f"Embedding {len(texts)} texts with all-MiniLM-L6-v2...")
embeddings = SentenceTransformerEmbedder._embed_texts(texts, embedding_dim=384, model_name="all-MiniLM-L6-v2")
for row, emb in zip(all_rows, embeddings):
    row["embedding"] = emb

# Rank by cosine similarity (vectors are already unit-normalised → dot product = cosine sim)
corpus_matrix = np.array([r["embedding"] for r in corpus], dtype=np.float32)
query_matrix  = np.array([r["embedding"] for r in queries], dtype=np.float32)
sims = query_matrix @ corpus_matrix.T  # shape: (n_queries, n_corpus)

corpus_ids = [r["doc_id"] for r in corpus]
query_relevant = {r["doc_id"]: set(r["relevant_doc_ids"]) for r in queries}
query_ranked   = {
    queries[i]["doc_id"]: [corpus_ids[j] for j in (-sims[i]).argsort()]
    for i in range(len(queries))
}

print(f"\nRecall@1:  {mean_recall_at_k(query_relevant, query_ranked, k=1):.4f}")
print(f"Recall@5:  {mean_recall_at_k(query_relevant, query_ranked, k=5):.4f}")
print(f"Recall@10: {mean_recall_at_k(query_relevant, query_ranked, k=10):.4f}")

Embedding 5483 texts with all-MiniLM-L6-v2...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Recall@1:  0.4823
Recall@5:  0.7379
Recall@10: 0.7833


---
## 2 — Full mloda Pipeline with FAISS

The direct pipeline above skips chunking and indexing. The production pipeline looks like:

```
eval_docs
  └── __chunked        (FixedSizeChunker:    split docs into 512-char chunks)
        └── __deduped  (ExactHashDeduplicator: remove exact duplicate chunks)
              └── __embedded  (SentenceTransformerEmbedder: 384-d vectors)
                    └── __indexed    (FaissFlatIndexer: build FAISS index)
                          └── __evaluated  (FaissRetrievalEvaluator: Recall@K)
```

mloda resolves each `__` segment to a `FeatureGroup`, wires the dependencies, and executes the chain. You declare what you want — mloda figures out how to compute it.

In [4]:
from mloda.user import mlodaAPI, PluginCollector, Feature, Options as MlodaOptions
from mloda_plugins.compute_framework.base_implementations.python_dict.python_dict_framework import PythonDictFramework

from rag_integration.feature_groups.datasets.text.scifact import ScifactDatasetSource
from rag_integration.feature_groups.rag_pipeline.chunking.fixed_size import FixedSizeChunker
from rag_integration.feature_groups.rag_pipeline.deduplication.exact_hash import ExactHashDeduplicator
from rag_integration.feature_groups.rag_pipeline.embedding.sentence_transformer import SentenceTransformerEmbedder
from rag_integration.feature_groups.rag_pipeline.vector_store.faiss_flat import FaissFlatIndexer
from rag_integration.feature_groups.evaluation.faiss_retrieval_evaluator import FaissRetrievalEvaluator

feature_name = "eval_docs__chunked__deduped__embedded__indexed__evaluated"

feature = Feature(
    feature_name,
    options=MlodaOptions(
        {ScifactDatasetSource.DATA_DIR: SCIFACT_DIR},  # group → propagates to root feature
        context={
            "chunking_method":      "fixed_size",
            "deduplication_method": "exact_hash",
            "keep_strategy":        "all_unique",
            "embedding_method":     "sentence_transformer",
            "model_name":           "all-MiniLM-L6-v2",
            "index_method":         "flat",
        },
    ),
)

print(f"Running pipeline: {feature_name}")
print("This will chunk ~5,183 docs, deduplicate, embed, index, and evaluate...")

raw_result = mlodaAPI.run_all(
    features=[feature],
    compute_frameworks={PythonDictFramework},
    plugin_collector=PluginCollector.enabled_feature_groups({
        ScifactDatasetSource,
        FixedSizeChunker,
        ExactHashDeduplicator,
        SentenceTransformerEmbedder,
        FaissFlatIndexer,
        FaissRetrievalEvaluator,
    }),
)

result_rows = raw_result[0]
if result_rows and isinstance(result_rows[0], list):
    result_rows = result_rows[0]

m = result_rows[0].get(feature_name, result_rows[0])
print(f"\nCorpus chunks: {m['num_corpus']}  |  Queries: {m['num_queries']}")
print(f"Recall@1:  {m['recall@1']:.4f}")
print(f"Recall@5:  {m['recall@5']:.4f}")
print(f"Recall@10: {m['recall@10']:.4f}")

Running pipeline: eval_docs__chunked__deduped__embedded__indexed__evaluated
This will chunk ~5,183 docs, deduplicate, embed, index, and evaluate...


  0%|          | 0/5183 [00:00<?, ?it/s]


Corpus chunks: 18888  |  Queries: 300
Recall@1:  0.5057
Recall@5:  0.7385
Recall@10: 0.7707


---
## 3 — Multimodal Retrieval: Flickr30K + CLIP

CLIP (Contrastive Language-Image Pre-Training) projects images and text into the **same 512-d embedding space**. This means you can retrieve images using a text query — no image labels needed.

Flickr30K test split: 1,000 images, each with 5 human-written captions → 5,000 text-to-image queries.

> **Note:** Set `FLICKR_DIR` at the top of this notebook. If the path doesn't exist, this section is skipped.

In [5]:
import os
if not os.path.exists(FLICKR_DIR):
    print(f"Flickr30K not found at {FLICKR_DIR!r} — set FLICKR_DIR above to run this section.")
else:
    from rag_integration.feature_groups.datasets.image.flickr30k import Flickr30kDatasetSource
    from rag_integration.feature_groups.image_pipeline.embedding.clip import CLIPImageEmbedder

    MAX_IMAGES = 100  # reduce for a quick run; use None for the full 1,000-image test split

    flickr_options = Options(context={
        Flickr30kDatasetSource.DATA_DIR: FLICKR_DIR,
        Flickr30kDatasetSource.MAX_SAMPLES: MAX_IMAGES,
    })
    flickr_rows = Flickr30kDatasetSource._load_dataset(flickr_options)

    images   = [r for r in flickr_rows if r["row_type"] == "corpus"]
    captions = [r for r in flickr_rows if r["row_type"] == "query"]
    print(f"Images: {len(images)}  |  Caption queries: {len(captions)}")

    model_path = CLIPImageEmbedder._resolve_model_path("openai/clip-vit-base-patch32")

    print("Embedding images (vision encoder)...")
    for row in images:
        row["embedding"] = CLIPImageEmbedder._embed_image(
            row.get("image_data") or b"", embedding_dim=512, model_name=model_path
        )

    print("Embedding captions (text encoder)...")
    for row in captions:
        row["embedding"] = CLIPImageEmbedder._embed_text(
            row.get("caption") or "", embedding_dim=512, model_name=model_path
        )

    # Cross-modal ranking: caption query → find most similar image
    img_matrix = np.array([r["embedding"] for r in images],   dtype=np.float32)
    cap_matrix = np.array([r["embedding"] for r in captions], dtype=np.float32)
    sims = cap_matrix @ img_matrix.T

    image_ids = [r["image_id"] for r in images]
    query_relevant = {r["image_id"]: set(r["relevant_image_ids"]) for r in captions}
    query_ranked   = {
        captions[i]["image_id"]: [image_ids[j] for j in (-sims[i]).argsort()]
        for i in range(len(captions))
    }

    print(f"\nRecall@1:  {mean_recall_at_k(query_relevant, query_ranked, k=1):.4f}")
    print(f"Recall@5:  {mean_recall_at_k(query_relevant, query_ranked, k=5):.4f}")
    print(f"Recall@10: {mean_recall_at_k(query_relevant, query_ranked, k=10):.4f}")
    print("\n(CLIP paper baseline on full test split: R@1≈0.65, R@5≈0.87, R@10≈0.92)")

Images: 100  |  Caption queries: 500
Embedding images (vision encoder)...


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Embedding captions (text encoder)...

Recall@1:  0.8540
Recall@5:  0.9700
Recall@10: 0.9880

(CLIP paper baseline on full test split: R@1≈0.65, R@5≈0.87, R@10≈0.92)


---
## What's next

### Swap in your own embedder
Subclass `BaseEmbedder` and add it to the `PluginCollector`. The chain wiring is unchanged.
```python
class MyEmbedder(BaseEmbedder):
    PROPERTY_MAPPING = {BaseEmbedder.EMBEDDING_METHOD: {"my_model": "...", ...}}
    def _embed_texts(cls, texts, embedding_dim, model_name): ...
```

### Add a new dataset
Subclass `BaseTextDatasetSource` (or `BaseImageDatasetSource`) and implement `_load_dataset`.
The source must return rows with `{"doc_id", "text", "row_type", "relevant_doc_ids"}`.

### Coming soon
- **PII redaction** stage (Presidio, regex, pattern-based) — slot in before embedding
- **LLM response generation** — full RAG answer with Claude
- **HNSW / IVF indexers** — approximate search for large corpora